# Huấn luyện mô hình trên Colab

Chạy lần lượt từng ô. Kết quả được ghi thẳng vào thư mục dự án trên Google Drive.

## 1. Kết nối Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Vào thư mục dự án

Sửa PROJECT_DIR nếu thư mục dự án nằm ở chỗ khác trong Drive.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/coffee-leaf-ml"
os.chdir(PROJECT_DIR)
print(os.listdir("."))
print(os.path.exists("data/processed/features_train.npz"))

## 3. Cài đặt

Ghim đúng bản scikit-learn đang dùng trên máy để mô hình lưu ra mở lại được ở app.

In [ ]:
!pip install -q scikit-learn==1.9.1
!pip install -q -e .

import sklearn
print(sklearn.__version__)

## 4. Huấn luyện

Tìm tham số bằng cross-validation 5 fold trên tập train. Không dùng tập test.

In [ ]:
!python scripts/train.py

## 5. Xem kết quả

In [ ]:
import json
import pandas as pd

cv_results = pd.read_csv("models/artifacts/cv_results.csv")
display(cv_results)
cv_per_class = pd.read_csv("models/artifacts/cv_per_class.csv")
display(cv_per_class)
cv_confusion_matrix = pd.read_csv("models/artifacts/cv_confusion_matrix.csv", index_col=0)
display(cv_confusion_matrix)

with open("models/artifacts/best_model_info.json") as info_file:
    print(json.load(info_file))

## 6. Biểu đồ kết quả

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARTIFACT_DIR = "models/artifacts"
FIGURE_DIR = "models/artifacts/figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

cv_results = pd.read_csv(f"{ARTIFACT_DIR}/cv_results.csv")
confusion_matrix_table = pd.read_csv(f"{ARTIFACT_DIR}/cv_confusion_matrix.csv", index_col=0)
per_class_table = pd.read_csv(f"{ARTIFACT_DIR}/cv_per_class.csv")

### Điểm train và điểm validation

In [ ]:
import sys

sys.path.insert(0, "src")
from coffee_leaf.evaluation import plots

cv_results = pd.read_csv(f"{ARTIFACT_DIR}/cv_results.csv")
figure = plots.plot_validation_scores(cv_results)
figure.savefig(f"{FIGURE_DIR}/train_vs_validation.png", dpi=150)
plt.show()

### Ma trận nhầm lẫn

In [ ]:
confusion_matrix_table = pd.read_csv(f"{ARTIFACT_DIR}/cv_confusion_matrix.csv", index_col=0)
figure = plots.plot_confusion_matrices(confusion_matrix_table)
figure.savefig(f"{FIGURE_DIR}/confusion_matrix.png", dpi=150)
plt.show()

### Kết quả theo từng lớp

In [ ]:
per_class_table = pd.read_csv(f"{ARTIFACT_DIR}/cv_per_class.csv")
figure = plots.plot_per_class_scores(per_class_table)
figure.savefig(f"{FIGURE_DIR}/per_class_scores.png", dpi=150)
plt.show()

In [ ]:
print(os.listdir("models/artifacts"))
print(os.listdir("models/artifacts/figures"))

## 7. Đánh giá trên tập test

Chỉ chạy khi mô hình đã chốt. Tập test chỉ dùng để báo cáo, không dùng để chọn lại mô hình.

In [ ]:
!python scripts/evaluate.py

In [ ]:
import pandas as pd

test_results = pd.read_csv("models/artifacts/test_results.csv")
display(test_results)
test_per_class = pd.read_csv("models/artifacts/test_per_class.csv")
display(test_per_class)
test_confusion_matrix = pd.read_csv("models/artifacts/test_confusion_matrix.csv", index_col=0)
display(test_confusion_matrix)

## 8. Độ quan trọng đặc trưng của cây

Phần này giải thích cây được chọn theo điểm CV trên tập train, chỉ dùng kết quả huấn luyện và metadata của tập train.

In [ ]:
!python scripts/feature_importance.py

In [ ]:
group_importance = pd.read_csv(f"{ARTIFACT_DIR}/tree_importance_by_group.csv")
region_importance = pd.read_csv(f"{ARTIFACT_DIR}/tree_importance_by_region.csv")
group_region_importance = pd.read_csv(f"{ARTIFACT_DIR}/tree_importance_by_group_region.csv")
display(group_importance)
display(region_importance)
display(group_region_importance)

In [ ]:
feature_importance_table = pd.read_csv(f"{ARTIFACT_DIR}/tree_feature_importance.csv")
group_importance = pd.read_csv(f"{ARTIFACT_DIR}/tree_importance_by_group.csv")
region_importance = pd.read_csv(f"{ARTIFACT_DIR}/tree_importance_by_region.csv")
figure = plots.plot_tree_importance(feature_importance_table, group_importance, region_importance)
figure.savefig(f"{FIGURE_DIR}/tree_feature_importance.png", dpi=150)
plt.show()